In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [21]:
df = pd.read_csv("fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [22]:
df.shape

(6000, 785)

In [30]:
x = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [31]:
xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.3)

In [32]:
xTrain = xTrain/255.00
xTest = xTest/255.00

In [33]:
xTrain

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4200, 784))

In [34]:
import torch
from torch.utils.data import Dataset, DataLoader

In [35]:
class CustomData(Dataset):

    def __init__(self, features, label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [36]:
trainData = CustomData(xTrain, yTrain)

In [37]:
testData = CustomData(xTest, yTest)

In [38]:
trainDataLoader = DataLoader(trainData, batch_size=64, shuffle=True)
testDataLoader = DataLoader(testData, batch_size=64, shuffle=False)

In [39]:
import torch.nn as nn

In [83]:
class NeuralNetwork(nn.Module):

    def __init__(self, feature_len):
        super().__init__()

        self.linear = nn.Sequential(
            nn.Linear(feature_len, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(32, 10)
        )

    def forward(self, features):
        predict = self.linear(features)
        return predict

In [84]:
epochs = 50
learning_rate = 0.01

In [90]:
model = NeuralNetwork(xTrain.shape[1])
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-2)

In [91]:
for epoch in range(epochs):
    for feature, label in trainDataLoader:
        y_pred = model(feature)
        loss = loss_fn(y_pred, label)
        optim.zero_grad()
        loss.backward()
        optim.step()
    print(f"Epochs: {epoch + 1}, Loss: {loss.item()}")

Epochs: 1, Loss: 1.4945085048675537
Epochs: 2, Loss: 1.2846969366073608
Epochs: 3, Loss: 1.1370773315429688
Epochs: 4, Loss: 1.0044887065887451
Epochs: 5, Loss: 0.8939038515090942
Epochs: 6, Loss: 0.9977931976318359
Epochs: 7, Loss: 0.8876892924308777
Epochs: 8, Loss: 0.7085624933242798
Epochs: 9, Loss: 0.6426571607589722
Epochs: 10, Loss: 0.6679470539093018
Epochs: 11, Loss: 0.6645478010177612
Epochs: 12, Loss: 0.5046156048774719
Epochs: 13, Loss: 0.48872485756874084
Epochs: 14, Loss: 0.45558589696884155
Epochs: 15, Loss: 0.5049456357955933
Epochs: 16, Loss: 0.685477614402771
Epochs: 17, Loss: 0.46181154251098633
Epochs: 18, Loss: 0.6373037695884705
Epochs: 19, Loss: 0.4343384802341461
Epochs: 20, Loss: 0.512891411781311
Epochs: 21, Loss: 0.34920090436935425
Epochs: 22, Loss: 0.48415952920913696
Epochs: 23, Loss: 0.3393200933933258
Epochs: 24, Loss: 0.5181646943092346
Epochs: 25, Loss: 0.43588727712631226
Epochs: 26, Loss: 0.4223055839538574
Epochs: 27, Loss: 0.40926599502563477
Epoch

In [92]:
model.eval()

NeuralNetwork(
  (linear): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=False)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [70]:
x = torch.randn(3, 784)

In [71]:
x.shape

torch.Size([3, 784])

In [72]:
with torch.no_grad():
    output = model(x)

In [73]:
output

tensor([[-9.0169, -4.8974,  8.5070,  1.8689,  1.5981, 10.5072, -1.4122, -0.9740,
         -2.7947, -7.2798],
        [ 2.4286, -1.7804,  1.3239,  4.1688, -5.5828, 10.0204, -4.8273,  0.8804,
         -8.1443, -1.2368],
        [-4.4408,  0.0316,  3.3365,  8.5617,  5.4843,  3.9098, -8.7895, -2.2933,
         -5.5735, -7.6275]])

In [74]:
torch.max(output, 1)

torch.return_types.max(
values=tensor([10.5072, 10.0204,  8.5617]),
indices=tensor([5, 5, 3]))

In [93]:
total = 0
correct = 0

with torch.no_grad():
    for feature, label in testDataLoader:
        model_output = model(feature)
        _, prediction = torch.max(model_output, 1)
        total += label.shape[0]
        correct += (prediction == label).sum().item()
print(correct/total)

0.8294444444444444


In [94]:
total = 0
correct = 0

with torch.no_grad():
    for feature, label in trainDataLoader:
        model_output = model(feature)
        _, prediction = torch.max(model_output, 1)
        total += label.shape[0]
        correct += (prediction == label).sum().item()
print(correct/total)

0.9478571428571428
